# Data Generation Stokes

In [ ]:
import os
import numpy as np
import torch
from joblib import Parallel, delayed
from tqdm import tqdm

from podcnf.DataGenerationStokes import ADR, mesh, Vh

In [ ]:
# # Creiamo la cartella per i dati grezzi se non esiste
# os.makedirs('../data/raw', exist_ok=True)

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
n_samples = 6400
np.random.seed(42)

eps_vals = np.random.uniform(0.01, 0.1, n_samples)
theta_vals = np.random.uniform(0, 2*np.pi, n_samples)
c1_vals = np.random.uniform(-1, 1, n_samples)
c2_vals = np.random.uniform(-1, 1, n_samples)
c3_vals = np.random.uniform(-1, 1, n_samples)

def compute_sample(i):
    return ADR(eps_vals[i], theta_vals[i], c1_vals[i], c2_vals[i], c3_vals[i])

In [ ]:
u_results = Parallel(n_jobs=2)(delayed(compute_sample)(i) for i in tqdm(range(N_SAMPLES)))

u = torch.tensor(np.array(u_results), dtype=torch.float32)

c1_tensor = torch.tensor(c1_vals, dtype=torch.float32).unsqueeze(1)
c2_tensor = torch.tensor(c2_vals, dtype=torch.float32).unsqueeze(1)
c3_tensor = torch.tensor(c3_vals, dtype=torch.float32).unsqueeze(1)
eps_tensor = torch.tensor(eps_vals, dtype=torch.float32).unsqueeze(1)
theta_tensor = torch.tensor(theta_vals, dtype=torch.float32).unsqueeze(1)

mu = torch.cat((c1_tensor, c2_tensor, c3_tensor), dim=1)

In [ ]:
data_to_save = {
    'u': u.cpu(),
    'eps': eps_tensor.cpu(),
    'theta': theta_tensor.cpu(),
    'mu': mu.cpu()
}

# torch.save(data_to_save, '../data/raw/stokes_dataset.pt')

# np.save('../data/raw/u_data.npy', u.numpy())
# np.save('../data/raw/mu_data.npy', mu.numpy())